# 2 — Fetch ATM Strike OHLC Data (Continuous Loop)

**Run frequency:** Every trading day, during market hours (09:15 – 15:30 IST)

**What this does:**
1. Logs in to Zerodha KiteConnect via automated TOTP
2. Starts a SparkSession
3. Reads instrument data from Parquet (written by Notebook 1)
4. Determines the ATM strike for BankNifty (live LTP or custom override)
5. Selects N strikes above/below ATM for the current expiry
6. Fetches 3-minute OHLC + OI history for each strike via KiteConnect API
7. Writes data to partitioned Parquet files
8. **Loops every minute** to keep pulling and appending the latest candle

**Runs in parallel with Notebook 3** (charts will auto-refresh as new data arrives)

---
### ⚙️ Configuration
Edit the `CONFIG` cell below to adjust parameters.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `BANKNIFTY` | `True` | Pull BankNifty data |
| `NIFTY` | `False` | Pull Nifty data |
| `CUSTOM_STRIKE` | `0` | Override ATM strike (0 = live LTP) |
| `NUM_STRIKES` | `9` | N strikes each side of ATM |
| `PULL_NEXT_EXPIRY` | `False` | Also pull next weekly expiry |
| `NUM_DAYS_HISTORY` | `1` | Days of history per pull |
| `LOOP_INTERVAL_MIN` | `1` | Minutes between pulls |
| `END_HOUR` | `15` | Stop loop at this hour (24h, set `None` to run indefinitely) |
| `END_MINUTE` | `30` | Stop loop at this minute |

In [1]:
# # ── Setup ──────────────────────────────────────────────────────────────────
import sys, os
# sys.path.insert(0, os.path.abspath('..'))
# os.chdir('..')  # Run from repo root so DataFiles/ paths resolve correctly

In [2]:
# ── CONFIG — edit these values ─────────────────────────────────────────────
BANKNIFTY         = True     # Pull BankNifty options data
NIFTY             = False    # Pull Nifty options data
CUSTOM_STRIKE     = 55000    # 0 = live LTP; set e.g. 56500 to override ATM
NUM_STRIKES       = 9        # How many ITM/OTM strikes each side of ATM
PULL_NEXT_EXPIRY  = False    # Also pull next weekly expiry
NUM_DAYS_HISTORY  = 1        # Calendar days of history per pull
LOOP_INTERVAL_MIN = 1        # Loop frequency in minutes
INTERVAL          = '3minute'

# Time-based loop exit — set to None to run indefinitely
END_HOUR          = 15       # 24-hour format (e.g. 15 = 3 PM)
END_MINUTE        = 33       # Minute (e.g. 30 = :30)

In [3]:
# ── Step 1: Login ──────────────────────────────────────────────────────────
from utils.kite_helpers import kite_login, get_spark_session

kite, kws, access_token = kite_login()
print('✅ Login successful')


2026-03-12 20:46:20,415 [INFO] utils.kite_helpers — Logging in as user: ZB4988
2026-03-12 20:46:20,627 [INFO] utils.kite_helpers — Generated TOTP pin for 2FA
2026-03-12 20:46:20,802 [INFO] utils.kite_helpers — Login successful — request_token: 02oXzjBlwcYrLZFTZdiCE4EO3qV45XOj
2026-03-12 20:46:21,027 [INFO] utils.kite_helpers — KiteConnect session established for user: ZB4988


✅ Login successful


In [4]:
# ── Step 2: Start SparkSession ─────────────────────────────────────────────
spark = get_spark_session(app_name='2_Fetch_Strikes_Data')
print('✅ Spark session ready')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/12 20:46:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/12 20:46:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✅ Spark session ready


In [6]:
import warnings
warnings.filterwarnings("ignore", message=".*fork.*")
warnings.filterwarnings("ignore", message=".*To enable non-built-in garbage collector.*")


In [7]:
# ── Step 3: Run continuous data pull loop ──────────────────────────────────
# This cell blocks until END_HOUR:END_MINUTE or Kernel → Interrupt
from utils.data_fetcher import run_data_loop

run_data_loop(
    kite                  = kite,
    spark                 = spark,
    banknifty             = BANKNIFTY,
    nifty                 = NIFTY,
    custom_strike         = CUSTOM_STRIKE,
    num_strikes           = NUM_STRIKES,
    pull_next_expiry      = PULL_NEXT_EXPIRY,
    num_days_history      = NUM_DAYS_HISTORY,
    interval              = INTERVAL,
    loop_interval_minutes = LOOP_INTERVAL_MIN,
    end_hour              = END_HOUR,
    end_minute            = END_MINUTE,
)

Running data pull at 2026-03-12 20:48:00


2026-03-12 20:48:01,690 [INFO] utils.data_fetcher — Data pull | BNF ATM=55000 | NIFTY ATM=0 | from=2026-03-11 20:48:01.690848 | to=2026-03-12 20:48:01.690848


BankNifty ATM Strike = 55000 | Nifty ATM Strike = 0 | Data pulled at: 2026-03-12 20:48:01.690848
Pulling Current Expiry (2026-03-30) — 38 tokens in parallel…


2026-03-12 20:48:06,213 [INFO] utils.data_fetcher — Parallel fetch done: 38 OK, 0 failed in 4.0s


Current Expiry data complete — 38 tokens in 45.0s
✅ Data pull completed.
